In [99]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [100]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(root="data", train=True, download=True, transform=ToTensor())
# Download test data from open datasets.
test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform=ToTensor())

In [101]:
batch_size = 64
# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [102]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [103]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        hidden_neurons = 512
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, hidden_neurons),
            nn.ReLU(),
            nn.Linear(hidden_neurons, hidden_neurons),
            nn.ReLU(),
            nn.Linear(hidden_neurons, 10)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [104]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [105]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [106]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss = 0
    correct = 0

    model.eval()
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    
    test_loss /= num_batches
    correct /= size

    print(f"Test error:\n    - Accuracy: {(100 * correct):>0.1f}%\n    - Avg loss: {test_loss:>8f}\n")

In [107]:
epochs = 5

for epoch in range(epochs):
    print(f"------ EPOCH {epoch + 1} ------\n")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)

------ EPOCH 1 ------

loss: 2.300783  [   64/60000]
loss: 2.289116  [ 6464/60000]
loss: 2.262539  [12864/60000]
loss: 2.267584  [19264/60000]
loss: 2.243503  [25664/60000]
loss: 2.213257  [32064/60000]
loss: 2.227678  [38464/60000]
loss: 2.183031  [44864/60000]
loss: 2.186188  [51264/60000]
loss: 2.160023  [57664/60000]
Test error:
    - Accuracy: 47.2%
    - Avg loss: 2.144364

------ EPOCH 2 ------

loss: 2.158020  [   64/60000]
loss: 2.150862  [ 6464/60000]
loss: 2.082349  [12864/60000]
loss: 2.110899  [19264/60000]
loss: 2.051282  [25664/60000]
loss: 1.982924  [32064/60000]
loss: 2.023973  [38464/60000]
loss: 1.932455  [44864/60000]
loss: 1.957563  [51264/60000]
loss: 1.876731  [57664/60000]
Test error:
    - Accuracy: 52.1%
    - Avg loss: 1.869905

------ EPOCH 3 ------

loss: 1.914223  [   64/60000]
loss: 1.883305  [ 6464/60000]
loss: 1.756891  [12864/60000]
loss: 1.808728  [19264/60000]
loss: 1.688111  [25664/60000]
loss: 1.637218  [32064/60000]
loss: 1.671851  [38464/60000]
l

In [108]:
model_save_path = "model.pth"

torch.save(model.state_dict(), model_save_path)
print(f"Saved PyTorch Model State to {model_save_path}")

Saved PyTorch Model State to model.pth


In [109]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load(model_save_path, weights_only=True))

<All keys matched successfully>

In [110]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]

with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
